# LeetCode 944 — Delete Columns to Make Sorted bằng Python

**Link đề:** https://leetcode.com/problems/delete-columns-to-make-sorted/
**Độ khó:** Easy · **Chủ đề:** Array, String (ma trận ký tự / lưới)

```python
class Solution:
    def minDeletionSize(self, strs: list[str]) -> int:
        ...
```

## Mục tiêu học
1. Nhìn danh sách chuỗi cùng độ dài như một **lưới** (grid) `n` hàng × `m` cột; truy cập `strs[r][c]`.
2. Hiểu "cột được sắp xếp theo thứ tự từ điển" = các ký tự từ trên xuống **không giảm**.
3. Đi từ cách dựng từng cột rồi so với bản sắp xếp → so sánh **cặp hàng kề nhau** có dừng sớm → **chuyển vị** `zip(*strs)` gọn gàng.
4. Nắm so sánh ký tự trong Python, `sorted` trên chuỗi, `zip(*...)`, `any`/`all`.
5. Trace, chứng minh, Time/Space, lỗi kinh điển (duyệt nhầm hàng/cột, dùng `<` thay `<=`).

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài — đếm số cột không được sắp xếp

Cho mảng `strs` gồm `n` chuỗi **cùng độ dài**. Xếp mỗi chuỗi trên một dòng, ta được một lưới. Ví dụ `strs = ["abc", "bce", "cae"]`:

```text
abc
bce
cae
```

Cột 0 (`a, b, c`) và cột 2 (`c, e, e`) được sắp xếp; cột 1 (`b, c, a`) thì không → xóa cột 1.

Trả về **số cột phải xóa** (các cột không được sắp xếp theo thứ tự từ điển).

| `strs` | Output | Giải thích |
|---|:---:|---|
| `["cba","daf","ghi"]` | `1` | cột 1: `b, a, h` không sắp xếp |
| `["a","b"]` | `0` | chỉ 1 cột, đã sắp xếp |
| `["zyx","wvu","tsr"]` | `3` | cả 3 cột đều giảm |

**Constraints:** `n == strs.length`, `1 <= n <= 100`, `1 <= strs[i].length <= 1000`, chữ thường.

Chú ý: "sắp xếp" ở đây là **không giảm** — hai ký tự bằng nhau liên tiếp (như `e, e`) vẫn hợp lệ.

In [1]:
strs = ["cba", "daf", "ghi"]
for row in strs:
    print(" ".join(row))
print("-" * 5)
for c in range(len(strs[0])):
    col = [row[c] for row in strs]
    print(f"cột {c}: {col}")

c b a
d a f
g h i
-----
cột 0: ['c', 'd', 'g']
cột 1: ['b', 'a', 'h']
cột 2: ['a', 'f', 'i']


## 2. Nền tảng Python cần dùng

### 2.1. Lưới ký tự: `strs[r][c]`

- `strs[r]` là chuỗi ở **hàng** `r`.
- `strs[r][c]` là ký tự ở **hàng** `r`, **cột** `c`.
- Số hàng `n = len(strs)`, số cột `m = len(strs[0])`.

Duyệt theo **cột** nghĩa là cố định `c`, cho `r` chạy `0..n-1`.

In [2]:
strs = ["abc", "bce", "cae"]
n, m = len(strs), len(strs[0])
print("n hàng =", n, "| m cột =", m)
print("strs[2][1] =", strs[2][1], "(hàng 2, cột 1)")
for r in range(n):
    print(f"hàng {r}, cột 1 -> {strs[r][1]}")

n hàng = 3 | m cột = 3
strs[2][1] = a (hàng 2, cột 1)
hàng 0, cột 1 -> b
hàng 1, cột 1 -> c
hàng 2, cột 1 -> a


### 2.2. So sánh ký tự

Ký tự so sánh theo **mã Unicode** (`ord`). Với chữ thường: `'a' < 'b' < ... < 'z'`. Đây chính là thứ tự từ điển cho từng ký tự.

In [3]:
print("a" < "b", "c" <= "c", "z" > "y", "b" > "c")
print(ord("a"), ord("b"), ord("z"))

True True True False
97 98 122


### 2.3. `sorted` trên chuỗi / list ký tự

`sorted("bca")` trả về **list** `['a', 'b', 'c']` (không phải chuỗi). So sánh list với list: `['b','c','a'] == sorted(['b','c','a'])`?

In [4]:
col = ["b", "c", "a"]
print(sorted(col), col == sorted(col))
col2 = ["c", "e", "e"]
print(sorted(col2), col2 == sorted(col2))
print(sorted("bca"), "".join(sorted("bca")))

['a', 'b', 'c'] False
['c', 'e', 'e'] True
['a', 'b', 'c'] abc


### 2.4. `zip(*strs)` — chuyển vị lưới (hàng ↔ cột)

`*strs` "trải" list thành nhiều tham số: `zip(*["abc","bce","cae"])` = `zip("abc", "bce", "cae")`. `zip` lấy ký tự thứ nhất của mỗi chuỗi → **cột 0**, ký tự thứ hai → **cột 1**, ...

In [5]:
strs = ["abc", "bce", "cae"]
for c, col in enumerate(zip(*strs)):
    print(f"cột {c}: {col}")

cột 0: ('a', 'b', 'c')
cột 1: ('b', 'c', 'a')
cột 2: ('c', 'e', 'e')


### 2.5. `any` / `all` và `sum` trên điều kiện

- `any(...)`: có **ít nhất một** phần tử đúng.
- `all(...)`: **mọi** phần tử đúng.
- `sum(cond for ...)`: đếm số điều kiện đúng (vì `True == 1`).

In [6]:
print(any(x > 2 for x in [1, 2, 3]), all(x > 0 for x in [1, 2, 3]))
print(sum(x % 2 == 0 for x in [1, 2, 3, 4]))

True True
2


## 3. Cách trực giác — dựng từng cột, so với bản đã sắp xếp

Với mỗi cột `c`:
1. Gom các ký tự `strs[0][c], strs[1][c], ..., strs[n-1][c]` thành list `col`.
2. Nếu `col != sorted(col)` → cột không sắp xếp → đếm.

In [7]:
def min_deletion_sorted(strs: list[str]) -> int:
    n, m = len(strs), len(strs[0])
    count = 0
    for c in range(m):
        col = [strs[r][c] for r in range(n)]
        if col != sorted(col):
            count += 1
    return count

for strs in [["cba", "daf", "ghi"], ["a", "b"], ["zyx", "wvu", "tsr"]]:
    print(strs, "->", min_deletion_sorted(strs))

['cba', 'daf', 'ghi'] -> 1
['a', 'b'] -> 0
['zyx', 'wvu', 'tsr'] -> 3


### 3.1. Chi phí của cách trực giác

Mỗi cột: dựng list `O(n)` + sắp xếp `O(n log n)`. Tổng `O(m · n log n)` thời gian, `O(n)` bộ nhớ phụ. Với `n ≤ 100`, `m ≤ 1000` vẫn nhanh, nhưng **sắp xếp là thừa**: ta chỉ cần biết "đã sắp xếp chưa", không cần dãy sắp xếp.

## 4. Nút thắt và insight

**Insight:** một dãy **không giảm** ⇔ **mọi cặp kề nhau** thỏa `col[r-1] <= col[r]`. Kiểm tra chỉ cần `n - 1` phép so sánh, và có thể **dừng ngay** khi gặp cặp vi phạm đầu tiên (`col[r-1] > col[r]`).

Hơn nữa, ta không cần dựng list cột: so sánh trực tiếp `strs[r-1][c]` với `strs[r][c]`.

Ví dụ cột 1 của `["cba","daf","ghi"]`: `b, a, h` → cặp đầu tiên `b > a` đã vi phạm → không cần xem `a, h`.

In [8]:
col = ["b", "a", "h"]
for r in range(1, len(col)):
    print(f"so {col[r-1]!r} với {col[r]!r}: {'OK' if col[r-1] <= col[r] else 'VI PHẠM -> dừng'}")
    if col[r - 1] > col[r]:
        break

so 'b' với 'a': VI PHẠM -> dừng


## 5. Lời giải chính — so sánh hàng kề nhau, dừng sớm

**Mảnh 1:** `count = 0`.

**Mảnh 2:** duyệt **cột** `c` từ `0` đến `m - 1` (vòng ngoài).

**Mảnh 3:** duyệt **hàng** `r` từ `1` đến `n - 1` (vòng trong): nếu `strs[r-1][c] > strs[r][c]` → `count += 1` và `break` (cột này đã bị xóa, không cần xét tiếp).

**Mảnh 4:** `return count`.

In [9]:
def min_deletion_adjacent(strs: list[str]) -> int:
    n, m = len(strs), len(strs[0])
    count = 0
    for c in range(m):
        for r in range(1, n):
            if strs[r - 1][c] > strs[r][c]:
                count += 1
                break
    return count

for strs in [["cba", "daf", "ghi"], ["a", "b"], ["zyx", "wvu", "tsr"]]:
    print(strs, "->", min_deletion_adjacent(strs))

['cba', 'daf', 'ghi'] -> 1
['a', 'b'] -> 0
['zyx', 'wvu', 'tsr'] -> 3


`break` chỉ thoát **vòng trong** (vòng hàng), vòng ngoài tiếp tục sang cột kế tiếp — đúng điều ta cần.

## 6. TRACE từng bước

Ví dụ `strs = ["cba", "daf", "ghi"]` (n = 3, m = 3):

| Cột c | r | strs[r-1][c] | strs[r][c] | So sánh | Hành động | count |
|:---:|:---:|:---:|:---:|---|---|:---:|
| 0 | 1 | c | d | c ≤ d | tiếp | 0 |
| 0 | 2 | d | g | d ≤ g | hết cột → giữ | 0 |
| 1 | 1 | b | a | b > a | **xóa**, break | 1 |
| 2 | 1 | a | f | a ≤ f | tiếp | 1 |
| 2 | 2 | f | i | f ≤ i | hết cột → giữ | 1 |

Kết quả `1`. Chú ý ở cột 1, cặp `(a, h)` không cần xét.

In [10]:
def trace_min_deletion(strs):
    n, m = len(strs), len(strs[0])
    count = 0
    print("Lưới:")
    for row in strs:
        print("   ", " ".join(row))
    for c in range(m):
        deleted = False
        for r in range(1, n):
            a, b_ = strs[r - 1][c], strs[r][c]
            if a > b_:
                count += 1
                deleted = True
                print(f"  cột {c}: hàng {r-1}->{r}: {a!r} > {b_!r} -> XÓA (count={count})")
                break
            print(f"  cột {c}: hàng {r-1}->{r}: {a!r} <= {b_!r} -> ok")
        if not deleted:
            print(f"  cột {c}: giữ lại")
    print("Kết quả:", count)
    return count

trace_min_deletion(["cba", "daf", "ghi"])

Lưới:
    c b a
    d a f
    g h i
  cột 0: hàng 0->1: 'c' <= 'd' -> ok
  cột 0: hàng 1->2: 'd' <= 'g' -> ok
  cột 0: giữ lại
  cột 1: hàng 0->1: 'b' > 'a' -> XÓA (count=1)
  cột 2: hàng 0->1: 'a' <= 'f' -> ok
  cột 2: hàng 1->2: 'f' <= 'i' -> ok
  cột 2: giữ lại
Kết quả: 1


1

In [11]:
# Ví dụ có ký tự bằng nhau (vẫn hợp lệ) và một hàng duy nhất
trace_min_deletion(["abc", "bce", "cae"])
print()
trace_min_deletion(["xyz"])

Lưới:
    a b c
    b c e
    c a e
  cột 0: hàng 0->1: 'a' <= 'b' -> ok
  cột 0: hàng 1->2: 'b' <= 'c' -> ok
  cột 0: giữ lại
  cột 1: hàng 0->1: 'b' <= 'c' -> ok
  cột 1: hàng 1->2: 'c' > 'a' -> XÓA (count=1)
  cột 2: hàng 0->1: 'c' <= 'e' -> ok
  cột 2: hàng 1->2: 'e' <= 'e' -> ok
  cột 2: giữ lại
Kết quả: 1

Lưới:
    x y z
  cột 0: giữ lại
  cột 1: giữ lại
  cột 2: giữ lại
Kết quả: 0


0

Với **một hàng duy nhất**, vòng `range(1, 1)` rỗng → mọi cột đều giữ → `0`. Đúng: một ký tự luôn "đã sắp xếp".

## 7. Các lời giải khác và bảng so sánh

### 7.1. Chuyển vị bằng `zip(*strs)` + `list == sorted`

In [12]:
def min_deletion_zip_sorted(strs: list[str]) -> int:
    return sum(list(col) != sorted(col) for col in zip(*strs))

for strs in [["cba", "daf", "ghi"], ["a", "b"], ["zyx", "wvu", "tsr"]]:
    print(strs, "->", min_deletion_zip_sorted(strs))

['cba', 'daf', 'ghi'] -> 1
['a', 'b'] -> 0
['zyx', 'wvu', 'tsr'] -> 3


`col` từ `zip` là **tuple**; `sorted(col)` là **list**. `tuple != list` luôn `True` dù cùng phần tử! Vì vậy phải `list(col)` trước khi so (xem Lỗi 3 ở mục 12).

### 7.2. Chuyển vị + `any` trên cặp kề (dừng sớm, gọn)

In [13]:
def min_deletion_zip_pairs(strs: list[str]) -> int:
    count = 0
    for col in zip(*strs):
        if any(a > b_ for a, b_ in zip(col, col[1:])):
            count += 1
    return count

for strs in [["cba", "daf", "ghi"], ["a", "b"], ["zyx", "wvu", "tsr"]]:
    print(strs, "->", min_deletion_zip_pairs(strs))

['cba', 'daf', 'ghi'] -> 1
['a', 'b'] -> 0
['zyx', 'wvu', 'tsr'] -> 3


### 7.3. Ghép cột thành chuỗi và so với `"".join(sorted(...))`

Biến thể của 7.1 dùng so sánh **chuỗi** thay vì list — tránh bẫy tuple/list.

In [14]:
def min_deletion_join(strs: list[str]) -> int:
    count = 0
    for col in zip(*strs):
        s = "".join(col)
        if s != "".join(sorted(s)):
            count += 1
    return count

print(min_deletion_join(["cba", "daf", "ghi"]), min_deletion_join(["zyx", "wvu", "tsr"]))

1 3


### 7.4. Duyệt theo hàng, đánh dấu cột hỏng (thân thiện bộ nhớ đệm)

Duyệt từng **cặp hàng** kề nhau, với mỗi cột đánh dấu `bad[c] = True` nếu vi phạm. Truy cập theo hàng liền mạch (mỗi lần xét hai chuỗi liên tiếp). Không dừng sớm theo cột, nhưng có thể bỏ qua cột đã hỏng.

In [15]:
def min_deletion_row_scan(strs: list[str]) -> int:
    m = len(strs[0])
    bad = [False] * m
    for prev, cur in zip(strs, strs[1:]):
        for c in range(m):
            if not bad[c] and prev[c] > cur[c]:
                bad[c] = True
    return sum(bad)

print(min_deletion_row_scan(["cba", "daf", "ghi"]), min_deletion_row_scan(["zyx", "wvu", "tsr"]))

1 3


### 7.5. Bảng so sánh

`n` = số hàng, `m` = số cột (độ dài chuỗi).

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| Dựng cột + `sorted` | `O(m · n log n)` | `O(n)` | dễ nghĩ nhất |
| **Cặp hàng kề, dừng sớm** | `O(n · m)` xấu nhất | `O(1)` | nên nộp |
| `zip(*)` + `list == sorted` | `O(m · n log n)` | `O(n·m)` (chuyển vị) | 1 dòng |
| `zip(*)` + `any` cặp kề | `O(n · m)` | `O(n·m)` | gọn, dừng sớm theo cột |
| Ghép chuỗi + sorted | `O(m · n log n)` | `O(n)` | tránh bẫy tuple/list |
| Duyệt theo hàng + `bad[]` | `O(n · m)` | `O(m)` | duyệt theo hàng |

## 8. Code nộp LeetCode

In [16]:
class Solution:
    def minDeletionSize(self, strs: list[str]) -> int:
        n, m = len(strs), len(strs[0])
        count = 0
        for c in range(m):
            for r in range(1, n):
                if strs[r - 1][c] > strs[r][c]:
                    count += 1
                    break
        return count

sol = Solution()
assert sol.minDeletionSize(["cba", "daf", "ghi"]) == 1
assert sol.minDeletionSize(["a", "b"]) == 0
assert sol.minDeletionSize(["zyx", "wvu", "tsr"]) == 3
print("Solution PASS 3 ví dụ đề bài")

Solution PASS 3 ví dụ đề bài


Bản ngắn kiểu Python cũng được chấp nhận:

```python
class Solution:
    def minDeletionSize(self, strs: list[str]) -> int:
        return sum(any(a > b for a, b in zip(col, col[1:])) for col in zip(*strs))
```

## 9. Vì sao thuật toán đúng?

**Mệnh đề:** một dãy `x_0, x_1, ..., x_{n-1}` không giảm ⇔ với mọi `r` từ 1 đến `n-1`: `x_{r-1} <= x_r`.

- (⇒) hiển nhiên từ định nghĩa không giảm.
- (⇐) tính bắc cầu: `x_0 <= x_1 <= ... <= x_{n-1}` nên mọi `i < j` có `x_i <= x_j`.

**Các cột độc lập:** việc xóa một cột không ảnh hưởng tới thứ tự của các cột khác (khác với LeetCode 955/960 — ở đó thứ tự giữa các **chuỗi** mới quan trọng). Vì vậy số cột phải xóa đúng bằng số cột **không** không-giảm, và không có cách nào xóa ít hơn.

**Dừng sớm đúng:** khi đã thấy một cặp vi phạm, cột chắc chắn phải xóa; xét thêm không đổi kết luận. ∎

## 10. Phân tích Time / Space

- **Time:** xấu nhất (mọi cột đều sắp xếp) phải so sánh `(n - 1) · m` cặp → `O(n · m)` — đúng bằng kích thước input, không thể tốt hơn vì phải đọc mọi ký tự để khẳng định cột sắp xếp. Tốt nhất (mọi cột hỏng ngay cặp đầu) chỉ `O(m)`.
- **Space:** `O(1)` phụ (vài biến chỉ số).

Với giới hạn `n ≤ 100`, `m ≤ 1000` → tối đa khoảng `10^5` phép so sánh.

In [17]:
import time, random
rng = random.Random(944)
n, m = 100, 1000
sorted_grid = ["".join(chr(97 + min(25, r // 4)) for _ in range(m)) for r in range(n)]   # mọi cột không giảm
random_grid = ["".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(m)) for _ in range(n)]
for name, grid in [("mọi cột sắp xếp", sorted_grid), ("ngẫu nhiên", random_grid)]:
    for f in [min_deletion_sorted, min_deletion_adjacent, min_deletion_zip_pairs]:
        t0 = time.perf_counter()
        r = f(grid)
        print(f"{name:16} {f.__name__:24} -> {r:4}, {time.perf_counter() - t0:.4f}s")

mọi cột sắp xếp  min_deletion_sorted      ->    0, 0.0041s
mọi cột sắp xếp  min_deletion_adjacent    ->    0, 0.0039s
mọi cột sắp xếp  min_deletion_zip_pairs   ->    0, 0.0049s
ngẫu nhiên       min_deletion_sorted      -> 1000, 0.0121s
ngẫu nhiên       min_deletion_adjacent    -> 1000, 0.0002s
ngẫu nhiên       min_deletion_zip_pairs   -> 1000, 0.0010s


`chr(97 + k)` đổi số thành chữ cái (`chr(97) == 'a'`). Lưới `sorted_grid` có hàng `r` toàn chữ `chr(97 + r // 4)` nên mọi cột không giảm.

## 11. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

Nhóm test:
1. Ba ví dụ đề.
2. Một hàng (mọi cột giữ); một cột.
3. Ký tự bằng nhau liên tiếp (không giảm vẫn hợp lệ).
4. Vi phạm ở **cặp cuối** của cột (kiểm tra không dừng quá sớm).
5. Ngẫu nhiên có seed, đối chiếu mọi cách.

In [18]:
tests = [
    (["cba", "daf", "ghi"], 1),
    (["a", "b"], 0),
    (["zyx", "wvu", "tsr"], 3),
    (["abc", "bce", "cae"], 1),
    (["xyz"], 0),
    (["a"], 0),
    (["b", "a"], 1),
    (["aaa", "aaa", "aaa"], 0),
    (["ab", "ab", "aa"], 1),          # cột 1 vi phạm ở cặp cuối: b > a
    (["rrjk", "furt", "guzm"], 2),
]
solvers = [min_deletion_sorted, min_deletion_adjacent, min_deletion_zip_sorted,
           min_deletion_zip_pairs, min_deletion_join, min_deletion_row_scan, Solution().minDeletionSize]
for strs, expected in tests:
    for f in solvers:
        got = f(strs)
        assert got == expected, (f.__name__, strs, got, expected)
print(f"PASS {len(tests)} test x {len(solvers)} cách")

PASS 10 test x 7 cách


In [19]:
for _ in range(500):
    n_ = rng.randint(1, 8); m_ = rng.randint(1, 8)
    strs = ["".join(rng.choice("abc") for _ in range(m_)) for _ in range(n_)]
    expected = min_deletion_sorted(strs)
    for f in solvers:
        assert f(strs) == expected, (f.__name__, strs)
print("PASS 500 lưới ngẫu nhiên")

PASS 500 lưới ngẫu nhiên


## 12. Lỗi người mới hay gặp

### Lỗi 1: Kiểm tra **hàng** thay vì **cột**

In [20]:
def wrong_rows(strs):
    return sum(list(s) != sorted(s) for s in strs)

print("wrong_rows(['cba','daf','ghi']) =", wrong_rows(["cba", "daf", "ghi"]), "| đúng: 1")

wrong_rows(['cba','daf','ghi']) = 2 | đúng: 1


**Vì sao sai:** đề hỏi **cột**. Mỗi chuỗi là một hàng. **Sửa:** cố định `c`, chạy `r`; hoặc dùng `zip(*strs)`.

### Lỗi 2: Dùng `>=` (coi ký tự bằng nhau là vi phạm)

In [21]:
def wrong_strict(strs):
    n, m = len(strs), len(strs[0])
    return sum(any(strs[r - 1][c] >= strs[r][c] for r in range(1, n)) for c in range(m))

print("wrong_strict(['abc','bce','cae']) =", wrong_strict(["abc", "bce", "cae"]), "| đúng: 1 (cột 2 'c,e,e' hợp lệ)")

wrong_strict(['abc','bce','cae']) = 2 | đúng: 1 (cột 2 'c,e,e' hợp lệ)


**Sửa:** vi phạm chỉ khi `strs[r-1][c] > strs[r][c]`.

### Lỗi 3: So sánh `tuple` từ `zip` với `list` từ `sorted`

In [22]:
col = ("a", "b", "c")
print("tuple == sorted(tuple)?", col == sorted(col), "<- luôn False vì khác kiểu")
print("list(col) == sorted(col)?", list(col) == sorted(col))

def wrong_tuple_vs_list(strs):
    return sum(col != sorted(col) for col in zip(*strs))

print("wrong_tuple_vs_list(['a','b']) =", wrong_tuple_vs_list(["a", "b"]), "| đúng: 0")

tuple == sorted(tuple)? False <- luôn False vì khác kiểu
list(col) == sorted(col)? True
wrong_tuple_vs_list(['a','b']) = 1 | đúng: 0


### Lỗi 4: Quên `break` → đếm một cột nhiều lần

In [23]:
def wrong_no_break(strs):
    n, m = len(strs), len(strs[0])
    count = 0
    for c in range(m):
        for r in range(1, n):
            if strs[r - 1][c] > strs[r][c]:
                count += 1             # thiếu break
    return count

print("wrong_no_break(['zyx','wvu','tsr']) =", wrong_no_break(["zyx", "wvu", "tsr"]), "| đúng: 3")

wrong_no_break(['zyx','wvu','tsr']) = 6 | đúng: 3


**Vì sao sai:** mỗi cột chỉ bị xóa **một lần**, dù có nhiều cặp vi phạm. **Sửa:** `break` sau khi đếm (hoặc dùng `any`).

### Lỗi 5: Dùng `range(n)` và đọc `strs[r+1]` → `IndexError`

Vòng so sánh cặp kề phải chạy `r` từ `1` (so `r-1` với `r`) hoặc từ `0` đến `n-2` (so `r` với `r+1`).

In [24]:
def wrong_index(strs):
    for c in range(len(strs[0])):
        for r in range(len(strs)):
            if strs[r][c] > strs[r + 1][c]:
                pass

try:
    wrong_index(["ab", "cd"])
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


## 13. Mẹo nhận dạng pattern

- Input là **danh sách chuỗi cùng độ dài** và đề nói "cột" → nghĩ tới **lưới** `strs[r][c]` hoặc **chuyển vị** `zip(*strs)`.
- "Kiểm tra dãy đã sắp xếp" → so sánh **cặp kề nhau**, không cần `sort`.
- "Đếm số X vi phạm" → vòng ngoài duyệt X, vòng trong tìm vi phạm đầu tiên rồi `break` (hoặc `any`).
- Chú ý **nghiêm ngặt** hay **không nghiêm ngặt** (`<` hay `<=`).

Bài liên quan: 955 Delete Columns to Make Sorted II (khó hơn: thứ tự giữa các chuỗi), 960 Delete Columns to Make Sorted III (LIS trên cột), 867 Transpose Matrix, 1572 Matrix Diagonal Sum.

## 14. Biến thể

### 14.1. Trả về **chỉ số** các cột cần xóa

In [25]:
def columns_to_delete(strs):
    return [c for c, col in enumerate(zip(*strs)) if any(a > b_ for a, b_ in zip(col, col[1:]))]

print(columns_to_delete(["cba", "daf", "ghi"]), columns_to_delete(["rrjk", "furt", "guzm"]))

[1] [0, 3]


### 14.2. Sau khi xóa, in lưới còn lại

In [26]:
def remaining_grid(strs):
    bad = set(columns_to_delete(strs))
    return ["".join(ch for c, ch in enumerate(row) if c not in bad) for row in strs]

print(remaining_grid(["rrjk", "furt", "guzm"]))

['rj', 'ur', 'uz']


### 14.3. Cột phải **tăng nghiêm ngặt** (không cho phép ký tự bằng nhau)

Chỉ đổi `>` thành `>=` — đây là lúc "Lỗi 2" trở thành đúng. Đọc kỹ đề!

In [27]:
def min_deletion_strict(strs):
    return sum(any(a >= b_ for a, b_ in zip(col, col[1:])) for col in zip(*strs))

print(min_deletion_strict(["abc", "bce", "cae"]))   # cột 1 (b,c,a) và cột 2 (c,e,e) -> 2

2


## 15. Bài tập tự luyện (TODO — không tự chạy)

### Bài 1 — Đoán output
```python
list(zip(*["ab", "cd", "ef"]))
("a", "b") == ["a", "b"]
sorted("dcba")
```

### Bài 2 — Kiểm tra một list ký tự có không giảm (không dùng `sorted`)

In [28]:
def exercise_is_non_decreasing(col) -> bool:
    # TODO: so sánh cặp kề
    pass

# assert exercise_is_non_decreasing(["a", "c", "c"]) == True
# assert exercise_is_non_decreasing(["b", "a"]) == False
# assert exercise_is_non_decreasing(["z"]) == True

### Bài 3 — Tự viết lời giải 944 (hai vòng lặp + break)

In [29]:
def exercise_min_deletion(strs: list[str]) -> int:
    # TODO
    pass

# assert exercise_min_deletion(["cba", "daf", "ghi"]) == 1
# assert exercise_min_deletion(["zyx", "wvu", "tsr"]) == 3

### Bài 4 — Transpose Matrix (LeetCode 867)

In [30]:
def exercise_transpose(matrix: list[list[int]]) -> list[list[int]]:
    # TODO: không dùng zip, dùng hai vòng lặp
    pass

# assert exercise_transpose([[1, 2, 3], [4, 5, 6]]) == [[1, 4], [2, 5], [3, 6]]

### Bài 5 — Lưới còn lại sau khi xóa (không dùng hàm ở mục 14)

In [31]:
def exercise_remaining(strs: list[str]) -> list[str]:
    # TODO
    pass

# assert exercise_remaining(["cba", "daf", "ghi"]) == ["ca", "df", "gi"]

### Bài 6 — Mini challenge: Delete Columns to Make Sorted II (LeetCode 955)

Lần này yêu cầu **các chuỗi** (hàng) sau khi xóa cột phải theo thứ tự từ điển `strs[0] <= strs[1] <= ...`. Trả về số cột ít nhất cần xóa. Gợi ý: duyệt cột từ trái sang phải, nhớ những cặp hàng kề **đã được phân định** (`strs[r-1] < strs[r]` nhờ một cột đã giữ).

In [32]:
def exercise_min_deletion_ii(strs: list[str]) -> int:
    # TODO
    pass

# assert exercise_min_deletion_ii(["ca", "bb", "ac"]) == 1
# assert exercise_min_deletion_ii(["xc", "yb", "za"]) == 0
# assert exercise_min_deletion_ii(["zyx", "wvu", "tsr"]) == 3

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | zip lấy ký tự cùng vị trí | tuple khác list | sorted trả list |
| 2 | cặp kề | `for i in range(1, len(col))` | `if col[i-1] > col[i]: return False` |
| 3 | cột ngoài, hàng trong | `break` sau khi đếm | xem mục 5 |
| 4 | kích thước mới `m × n` | `res[c][r] = matrix[r][c]` | khởi tạo `[[0]*n for _ in range(m)]` |
| 5 | tìm cột hỏng trước | `set` các cột hỏng | ghép ký tự ở cột tốt |
| 6 | mảng `sorted_pair[r]` | cột c gây vi phạm ở cặp chưa phân định → xóa | nếu giữ cột, cập nhật cặp có `strs[r-1][c] < strs[r][c]` |

## 17. Đáp án tham khảo

Bài 1: `[('a', 'c', 'e'), ('b', 'd', 'f')]`, `False`, `['a', 'b', 'c', 'd']`.

In [33]:
print(list(zip(*["ab", "cd", "ef"])), ("a", "b") == ["a", "b"], sorted("dcba"))

[('a', 'c', 'e'), ('b', 'd', 'f')] False ['a', 'b', 'c', 'd']


In [34]:
def answer_is_non_decreasing(col):
    for i in range(1, len(col)):
        if col[i - 1] > col[i]:
            return False
    return True

def answer_min_deletion(strs):
    count = 0
    for c in range(len(strs[0])):
        for r in range(1, len(strs)):
            if strs[r - 1][c] > strs[r][c]:
                count += 1
                break
    return count

def answer_transpose(matrix):
    n, m = len(matrix), len(matrix[0])
    res = [[0] * n for _ in range(m)]
    for r in range(n):
        for c in range(m):
            res[c][r] = matrix[r][c]
    return res

def answer_remaining(strs):
    keep = [c for c in range(len(strs[0])) if answer_is_non_decreasing([row[c] for row in strs])]
    return ["".join(row[c] for c in keep) for row in strs]

def answer_min_deletion_ii(strs):
    n = len(strs)
    decided = [False] * n          # decided[r]: strs[r-1] < strs[r] đã chắc chắn
    deleted = 0
    for c in range(len(strs[0])):
        if any(not decided[r] and strs[r - 1][c] > strs[r][c] for r in range(1, n)):
            deleted += 1           # giữ cột này sẽ phá thứ tự -> xóa
            continue
        for r in range(1, n):
            if strs[r - 1][c] < strs[r][c]:
                decided[r] = True
    return deleted

assert answer_is_non_decreasing(["a", "c", "c"]) and not answer_is_non_decreasing(["b", "a"]) and answer_is_non_decreasing(["z"])
assert answer_min_deletion(["cba", "daf", "ghi"]) == 1 and answer_min_deletion(["zyx", "wvu", "tsr"]) == 3
assert answer_transpose([[1, 2, 3], [4, 5, 6]]) == [[1, 4], [2, 5], [3, 6]]
assert answer_remaining(["cba", "daf", "ghi"]) == ["ca", "df", "gi"]
assert answer_min_deletion_ii(["ca", "bb", "ac"]) == 1
assert answer_min_deletion_ii(["xc", "yb", "za"]) == 0
assert answer_min_deletion_ii(["zyx", "wvu", "tsr"]) == 3
for strs, expected in tests:
    assert answer_min_deletion(strs) == expected
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


Ghi chú Bài 6: điểm khác biệt với 944 là các cột **không còn độc lập** — một cột đã giữ có thể "phân định" thứ tự giữa hai hàng, khiến các vi phạm ở cột sau trên cặp hàng đó không còn quan trọng.

## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def minDeletionSize(self, strs: list[str]) -> int:
        count = 0
        for c in range(len(strs[0])):          # mỗi cột
            for r in range(1, len(strs)):      # cặp hàng kề nhau
                if strs[r - 1][c] > strs[r][c]:
                    count += 1
                    break
        return count
```

**Câu thần chú:** *"Cột ngoài, hàng trong; thấy trên lớn hơn dưới là xóa cột rồi break."*

**Checklist trước khi submit:**
- [ ] Duyệt **cột** (không phải hàng).
- [ ] Vi phạm là `>` (ký tự bằng nhau vẫn hợp lệ).
- [ ] `break` sau khi đếm một cột.
- [ ] Vòng hàng bắt đầu từ `1` (so `r-1` với `r`).
- [ ] Nếu dùng `zip(*strs)` + `sorted`, nhớ đổi tuple sang list.